In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import contextlib
import io
import numpy as np
from datetime import date
from scipy.optimize import minimize
from sklearn.metrics import root_mean_squared_error, r2_score

from aquacrop import AquaCrop, Crop, Soil, Weather

from aquacrop_slovenia import config
from aquacrop_slovenia.plots import plot_yield_timeseries_comparison, plot_yield_scatter
from aquacrop_slovenia.reading_data import get_yield, get_yield_for_comparison, get_co2_for_aquacrop, get_station_weather
from aquacrop_slovenia.parameter_defaults_rakican import (
    rakican_soil_layers,
    rakican_curve_number,
    rakican_readily_evaporable_water,
    rakican_maize_params,
    rakican_optimal_management,
    rakican_initial_cond,
    rakican_groundwater
)

In [17]:
LOCATION = "rakican"
output_dir = config.RAMDISK_DIR / "calibration_rakican"

simulation_periods = [
    {
        "start_date": date(year, 1, 1),
        "end_date": date(year, 12, 31),
        "planting_date": date(year, 4, 20),
        "is_seeding_year": True,
    }
    for year in range(1993, 2024) if year not in [1998, 2017, 2023]
]

temperatures, eto, rain = load_station_weather(355)
historical_co2 = get_co2_for_aquacrop("historical")

weather = Weather(
    location=LOCATION,
    temperatures=temperatures,
    eto_values=eto,
    rainfall_values=rain,
    record_type=1,
    first_day=1,
    first_month=1,
    first_year=1993,
    co2_records=historical_co2,
)

soil = Soil(
    name=f"{LOCATION} soil",
    description=f"{LOCATION} loamy sand soil",
    soil_layers=rakican_soil_layers,
    curve_number=rakican_curve_number,
    readily_evaporable_water=rakican_readily_evaporable_water,
)

observed_yield = get_yield_for_comparison(LOCATION, "grain", "A", "N0")

In [18]:
# Parameters to calibrate: param_name -> (initial_value, lower_bound, upper_bound)
# Add or remove entries here to control what gets calibrated.
calibration_params = {
    "water_productivity":   (33.7, 28.0, 40.0),
    "harvest_index":        (0.48, 0.45, 0.60),
    "gdd_flowering":        (880, 750, 950),
    "gdd_flowering_length": (180, 120, 240),
    "max_canopy_cover":     (0.96, 0.85, 0.98),
    "max_rooting_depth":    (2.3, 1.5, 3.0),
    "veg_growth_impact_hi": (7.0, 5.0, 8.0),
    "stomatal_closure_impact_hi": (3.0, 2.0, 4.0),
    "dry_matter_content":   (85, 70, 95),
}

param_names = list(calibration_params.keys())
x0     = np.array([v[0] for v in calibration_params.values()])
bounds = [(v[1], v[2]) for v in calibration_params.values()]

# Parameters whose original value is int must stay int — AquaCrop's Fortran
# reader uses integer format for these fields and crashes if floats are used.
INTEGER_PARAMS = {
    name for name, val in rakican_maize_params.items() if isinstance(val, int)
}

print("Parameters to calibrate:")
for name, (init, lo, hi) in calibration_params.items():
    print(f"  {name}: initial={init}, bounds=[{lo}, {hi}]")

Parameters to calibrate:
  water_productivity: initial=33.7, bounds=[20.0, 45.0]
  harvest_index: initial=0.48, bounds=[0.4, 0.6]
  gdd_flowering: initial=880, bounds=[700, 1000]
  gdd_flowering_length: initial=180, bounds=[120, 240]
  max_canopy_cover: initial=0.96, bounds=[0.8, 0.99]
  max_rooting_depth: initial=2.3, bounds=[1.5, 3.0]
  veg_growth_impact_hi: initial=7.0, bounds=[5.0, 9.0]
  stomatal_closure_impact_hi: initial=3.0, bounds=[2.0, 4.0]
  dry_matter_content: initial=85, bounds=[70, 95]


In [19]:
iteration = [0]

def objective(x):
    iteration[0] += 1

    for j in range(len(x)):
        if param_names[j] in INTEGER_PARAMS:
            rakican_maize_params[param_names[j]] = int(round(x[j]))
        else:
            rakican_maize_params[param_names[j]] = x[j]

    crop = Crop(
        name=f"{LOCATION} maize",
        description=f"{LOCATION} maize (calibrating)",
        params=rakican_maize_params,
    )
    simulation = AquaCrop(
        simulation_periods=simulation_periods,
        crop=crop,
        soil=soil,
        management=rakican_optimal_management,
        initial_conditions=rakican_initial_conditions,
        climate=weather,
        working_dir=output_dir,
        need_daily_output=False,
        need_seasonal_output=True,
        need_harvest_output=False,
        need_evaluation_output=False,
    )

    try:
        with contextlib.redirect_stdout(io.StringIO()):
            results = simulation.run()
    except Exception as e:
        print(f"Iter {iteration[0]:4d}: FAILED — {type(e).__name__}: {e}")
        return 1e6

    seasonal = (
        results["season"][["Year1", "Y(dry)"]]
        .rename(columns={"Year1": "year", "Y(dry)": "yield_modeled"})
    )
    merged = observed_yield.merge(seasonal, on="year")
    rmse = root_mean_squared_error(merged["yield"], merged["yield_modeled"])

    vals = ", ".join(f"{n}={v:.4f}" for n, v in zip(param_names, x))
    print(f"Iter {iteration[0]:4d}: RMSE={rmse:.4f}  [{vals}]")

    return rmse

In [20]:
iteration[0] = 0

result = minimize(
    objective,
    x0,
    method="Nelder-Mead",
    bounds=bounds,
    options={
        "maxiter": 20,
        "xatol": 1e-2,
        "fatol": 1e-4,
        "adaptive": True,
    }
)

print(f"\nOptimization {'succeeded' if result.success else 'did not fully converge'}: {result.message}")
print(f"Iterations: {iteration[0]}")
print(f"Final RMSE: {result.fun:.4f}")
print("\nCalibrated parameters:")
for name, val in zip(param_names, result.x):
    print(f"  {name}: {val:.6f}  (initial: {calibration_params[name][0]})")

Iter    1: RMSE=3.8814  [water_productivity=33.7000, harvest_index=0.4800, gdd_flowering=880.0000, gdd_flowering_length=180.0000, max_canopy_cover=0.9600, max_rooting_depth=2.3000, veg_growth_impact_hi=7.0000, stomatal_closure_impact_hi=3.0000, dry_matter_content=85.0000]
Iter    2: RMSE=4.2779  [water_productivity=35.3850, harvest_index=0.4800, gdd_flowering=880.0000, gdd_flowering_length=180.0000, max_canopy_cover=0.9600, max_rooting_depth=2.3000, veg_growth_impact_hi=7.0000, stomatal_closure_impact_hi=3.0000, dry_matter_content=85.0000]
Iter    3: RMSE=4.2309  [water_productivity=33.7000, harvest_index=0.5040, gdd_flowering=880.0000, gdd_flowering_length=180.0000, max_canopy_cover=0.9600, max_rooting_depth=2.3000, veg_growth_impact_hi=7.0000, stomatal_closure_impact_hi=3.0000, dry_matter_content=85.0000]
Iter    4: RMSE=3.8814  [water_productivity=33.7000, harvest_index=0.4800, gdd_flowering=924.0000, gdd_flowering_length=180.0000, max_canopy_cover=0.9600, max_rooting_depth=2.3000, 

In [21]:
for j in range(len(result.x)):
        if param_names[j] in INTEGER_PARAMS:
            rakican_maize_params[param_names[j]] = int(round(result.x[j]))
        else:
            rakican_maize_params[param_names[j]] = result.x[j]

calibrated_crop = Crop(
    name=f"{LOCATION} maize calibrated",
    description=f"{LOCATION} maize calibrated",
    params=rakican_maize_params
)

final_simulation = AquaCrop(
    simulation_periods=simulation_periods,
    crop=calibrated_crop,
    soil=soil,
    management=rakican_optimal_management,
    initial_conditions=rakican_initial_conditions,
    climate=weather,
    working_dir=output_dir,
    need_daily_output=False,
    need_seasonal_output=True,
    need_harvest_output=False,
    need_evaluation_output=False,
)

final_results = final_simulation.run()
final_seasonal = final_results["season"]

print(root_mean_squared_error(observed_yield["yield"].values, final_seasonal.loc[:, "Y(dry)"].values))
print(r2_score(observed_yield["yield"].values, final_seasonal.loc[:, "Y(dry)"].values))

plot_yield_timeseries_comparison(final_seasonal, get_yield(LOCATION, "grain", "A", "N3"), "Y(dry)")
plot_yield_scatter(final_seasonal, get_yield(LOCATION, "grain", "A", "N3"))

TypeError: Crop.__init__() missing 2 required positional arguments: 'c_name' and 'planting_date'